# Real MPC/ORAM prototype for SLE

This notebook compiles and runs five MP-SPDZ parties. It replaces the trusted Python reconstruction used in the semantic notebook. The default smoke profile is intentionally small so the cryptographic wiring can be checked before any expensive experiment.

**Important:** MP-SPDZ compilation is CPU-intensive. Run this notebook in Google Colab. Do not execute its build cells on the local Windows PC.

In [ ]:
# CELL 1. Experiment profile
# The smoke profile tests correctness. The controlled profile is a larger
# cryptographic benchmark. The 100 x 1024 paper target should be attempted
# only after both profiles pass because recursive ORAM is expensive.

PROFILES = {
    'smoke': dict(n=8, root_bits=32, block_bits=4, capacity=4, threshold=8),
    'controlled': dict(n=32, root_bits=128, block_bits=4, capacity=8, threshold=32),
    'paper_target': dict(n=100, root_bits=1024, block_bits=8, capacity=8, threshold=256),
}
PROFILE_NAME = 'smoke'
CFG = PROFILES[PROFILE_NAME]
PLAYERS = 5
THRESHOLD = 2
print('Selected:', PROFILE_NAME, CFG)


## Install MP-SPDZ

This is the only long setup stage. It downloads the official MP-SPDZ source and builds the five-party Shamir runtime. It does not access CelebA or any face image.

In [ ]:
# CELL 2. Linux dependencies and MP-SPDZ build
import os, subprocess, time
from pathlib import Path

def run(command, cwd=None, env=None, log_path=None):
    print('$', command)
    started = time.perf_counter()
    process = subprocess.Popen(
        command, cwd=cwd, env=env, shell=True, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line)
    returncode = process.wait()
    output = ''.join(output_lines)
    elapsed = time.perf_counter() - started
    print(f'elapsed={elapsed:.2f}s returncode={returncode}')
    if log_path:
        Path(log_path).write_text(output, encoding='utf-8')
    if returncode:
        raise RuntimeError(command)
    return output, elapsed

# A previous interrupted Colab installation can leave dpkg in an
# unfinished state. Repair it before requesting new packages.
run('DEBIAN_FRONTEND=noninteractive dpkg --configure -a')
run('DEBIAN_FRONTEND=noninteractive apt-get -qq --fix-broken install -y')
run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y automake build-essential clang cmake git '
    'libboost-dev libboost-filesystem-dev libboost-iostreams-dev libboost-thread-dev '
    'libgmp-dev libntl-dev libsodium-dev libssl-dev libtool')
if not Path('/content/MP-SPDZ').exists():
    run('git clone --depth 1 https://github.com/data61/MP-SPDZ.git /content/MP-SPDZ')
run('make -j2 setup', cwd='/content/MP-SPDZ')
run('make -j2 shamir-party.x', cwd='/content/MP-SPDZ')
# MP-SPDZ encrypts communication between local party processes.
# Generate one certificate/key pair for each of the five parties.
run('Scripts/setup-ssl.sh 5', cwd='/content/MP-SPDZ')


## Materialize the reviewed protocol sources

The following cells place the exact checked `.mpc` programs and input helpers into the Colab filesystem.

In [ ]:
from pathlib import Path
target = Path('/content/sle_mpc_project/Programs/Source/sle_full_scan.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nReal MPC baseline for binary 1:N biometric identification.\n\nPlayer 0 supplies the enrolled binary roots, identity labels, and stable\ntie-breaking tags. Player 1 supplies one binary probe. All values become\nsecret shares on input. The five computing parties evaluate every Hamming\ndistance and the complete ArgMin without reconstructing a root, a distance,\nor an intermediate comparison. Only the final identity/reject decision is\nopened.\n\nCompile-time arguments:\n    sle_full_scan <n_records> <root_bits> <accept_threshold>\n"""\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 4:\n    raise CompilerError(\n        "Usage: sle_full_scan <n_records> <root_bits> <accept_threshold>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nACCEPT_THRESHOLD = int(program.args[3])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS < 8:\n    raise CompilerError("root_bits must be at least eight")\nif not 0 <= ACCEPT_THRESHOLD <= ROOT_BITS:\n    raise CompilerError("accept_threshold must be in [0, root_bits]")\n\n\n# The database owner inputs one label, one stable tie tag, and ROOT_BITS\n# binary values for every enrolled record. These arrays hold secret shares.\nlabels = sint.Array(N_RECORDS)\ntie_tags = sint.Array(N_RECORDS)\ngallery = sint.Matrix(N_RECORDS, ROOT_BITS)\n\nlabels.input_from(0)\ntie_tags.input_from(0)\ngallery.input_from(0)\n\n# The probe owner is independent from the database owner. Input conversion\n# secret-shares the probe among the computing parties.\nprobe = sint.Array(ROOT_BITS)\nprobe.input_from(1)\n\n\ndef secure_hamming(enrolled, query):\n    """Hamming distance for secret binary values in an arithmetic field."""\n\n    # For bits a,b, XOR(a,b) = a + b - 2ab. Every multiplication is handled\n    # by the selected MP-SPDZ protocol and does not open either bit.\n    return sum(\n        enrolled[j] + query[j] - 2 * enrolled[j] * query[j]\n        for j in range(ROOT_BITS)\n    )\n\n\nstart_timer(1)\n\n# ROOT_BITS + 1 is larger than every valid distance. Tie tags make the result\n# deterministic if two gallery entries have equal distance to the probe.\nbest_distance = sint(ROOT_BITS + 1)\nbest_label = sint(-1)\nbest_tie_tag = sint(2 ** 31 - 1)\n\nfor row in range(N_RECORDS):\n    distance = secure_hamming(gallery[row], probe)\n    strictly_closer = distance < best_distance\n    equal_distance = distance == best_distance\n    smaller_tie_tag = tie_tags[row] < best_tie_tag\n    better = strictly_closer + equal_distance * smaller_tie_tag\n\n    best_distance = better.if_else(distance, best_distance)\n    best_label = better.if_else(labels[row], best_label)\n    best_tie_tag = better.if_else(tie_tags[row], best_tie_tag)\n\naccepted = best_distance <= ACCEPT_THRESHOLD\nopened_decision = accepted.if_else(best_label, sint(-1)).reveal()\nopened_accept = accepted.reveal()\n\nstop_timer(1)\n\nprint_ln("SLE_MPC_RESULT mode=full_scan accepted=%s identity=%s",\n         opened_accept, opened_decision)\nprint_ln("SLE_MPC_PRIVATE distance=not_opened roots=not_opened")\n\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
from pathlib import Path
target = Path('/content/sle_mpc_project/Programs/Source/sle_oram_index.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nReal MPC-ORAM candidate retrieval for binary 1:N identification.\n\nTwo oblivious memories are used:\n  1. bucket_oram maps a secret LSH bucket address to fixed-size candidate IDs;\n  2. root_oram maps every secret candidate ID to its biometric root and label.\n\nThe physical access pattern is hidden by MP-SPDZ ORAM. Secure Hamming and\nArgMin then open only Top-1/reject. Small smoke tests use the linear ORAM\nbackend selected by OptimalORAM; larger tables automatically use recursive\ntree ORAM.\n\nCompile-time arguments:\n    sle_oram_index <n_records> <root_bits> <block_bits>\n                   <bucket_capacity> <accept_threshold>\n"""\n\nfrom oram import OptimalORAM\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 6:\n    raise CompilerError(\n        "Usage: sle_oram_index <n_records> <root_bits> <block_bits> "\n        "<bucket_capacity> <accept_threshold>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nACCEPT_THRESHOLD = int(program.args[5])\n\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\nif BLOCK_BITS < 1 or BLOCK_BITS > 8:\n    raise CompilerError("block_bits must be in [1, 8]")\nif BUCKET_CAPACITY < 1:\n    raise CompilerError("bucket_capacity must be positive")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nDUMMY_ROW = N_RECORDS\nROW_BITS = max(1, (N_RECORDS + 1).bit_length())\n\n\n# Player 0 initializes the persistent protected database. The dummy row lets\n# fixed-capacity buckets hide their true occupancy.\nlabels = sint.Array(N_RECORDS + 1)\nvalid_records = sint.Array(N_RECORDS + 1)\ntie_tags = sint.Array(N_RECORDS + 1)\ngallery = sint.Matrix(N_RECORDS + 1, ROOT_BITS)\n\nlabels.input_from(0)\nvalid_records.input_from(0)\ntie_tags.input_from(0)\ngallery.input_from(0)\n\n# The complete padded bucket table is secret input. Each cell contains a row\n# ID in [0,N_RECORDS] where N_RECORDS denotes the dummy record.\nbucket_rows = sint.Matrix(N_BUCKETS, BUCKET_CAPACITY)\nbucket_rows.input_from(0)\n\nprobe = sint.Array(ROOT_BITS)\nprobe.input_from(1)\n\n\n# Root ORAM values contain label, validity, tie tag, and every binary root bit.\nroot_entry_sizes = (ROW_BITS, 1, 32) + (1,) * ROOT_BITS\nroot_oram = OptimalORAM(\n    N_RECORDS + 1,\n    entry_size=root_entry_sizes,\n)\n\nfor row in range(N_RECORDS + 1):\n    root_oram[row] = tuple(\n        [labels[row], valid_records[row], tie_tags[row]]\n        + [gallery[row][j] for j in range(ROOT_BITS)]\n    )\n\n# Bucket ORAM values are fixed-length vectors of secret candidate row IDs.\nbucket_oram = OptimalORAM(\n    N_BUCKETS,\n    entry_size=(ROW_BITS,) * BUCKET_CAPACITY,\n)\n\nfor bucket in range(N_BUCKETS):\n    bucket_oram[bucket] = tuple(\n        bucket_rows[bucket][slot] for slot in range(BUCKET_CAPACITY)\n    )\n\n\ndef secure_hamming(root_values, query):\n    return sum(\n        root_values[j] + query[j] - 2 * root_values[j] * query[j]\n        for j in range(ROOT_BITS)\n    )\n\n\nstart_timer(2)\n\nbest_distance = sint(ROOT_BITS + 1)\nbest_label = sint(-1)\nbest_tie_tag = sint(2 ** 31 - 1)\nvalid_candidate_count = sint(0)\n\nfor block in range(N_BLOCKS):\n    # The query-derived address remains secret. ORAM hides which logical\n    # bucket and which physical storage path are touched.\n    bucket_value = sint(0)\n    for bit in range(BLOCK_BITS):\n        bucket_value += probe[block * BLOCK_BITS + bit] * (2 ** bit)\n    secret_bucket_address = bucket_value + block * BUCKET_VALUES\n\n    candidate_rows = bucket_oram[secret_bucket_address]\n    if BUCKET_CAPACITY == 1:\n        candidate_rows = (candidate_rows,)\n\n    for slot in range(BUCKET_CAPACITY):\n        candidate_row = candidate_rows[slot]\n        entry = root_oram[candidate_row]\n        label = entry[0]\n        valid = entry[1]\n        tie_tag = entry[2]\n        root = entry[3:]\n\n        distance = secure_hamming(root, probe)\n        strictly_closer = distance < best_distance\n        equal_distance = distance == best_distance\n        smaller_tie_tag = tie_tag < best_tie_tag\n        better = valid * (\n            strictly_closer + equal_distance * smaller_tie_tag\n        )\n\n        best_distance = better.if_else(distance, best_distance)\n        best_label = better.if_else(label, best_label)\n        best_tie_tag = better.if_else(tie_tag, best_tie_tag)\n        valid_candidate_count += valid\n\naccepted = (valid_candidate_count > 0) * (best_distance <= ACCEPT_THRESHOLD)\nopened_decision = accepted.if_else(best_label, sint(-1)).reveal()\nopened_accept = accepted.reveal()\n\nstop_timer(2)\n\nprint_ln("SLE_MPC_RESULT mode=oram accepted=%s identity=%s",\n         opened_accept, opened_decision)\nprint_ln("SLE_MPC_PRIVATE bucket_addresses=not_opened candidates=not_opened "\n         "distance=not_opened")\n\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
from pathlib import Path
target = Path('/content/sle_mpc_project/Programs/Source/sle_transition.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nMPC implementation of one Synchronized Locator Evolution transition.\n\nAll five parties contribute to the epoch key. The complete key is never an\ninput of one server. A MiMC-style field PRP evolves every database and index\ncopy of a locator. A random-challenge multiset fingerprint checks the hidden\n(locator, block, bucket) relation before the new state is selected. Only the\naccept/reject bit is opened; committed locators are written as secret shares.\n\nCompile-time arguments:\n    sle_transition <n_records> <root_bits> <block_bits>\n                   <bucket_capacity> <epoch> <fault_mode>\n\nfault_mode: 0 clean, 1 drop first valid index entry,\n            2 corrupt first valid locator, 3 corrupt first bucket relation.\n"""\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 7:\n    raise CompilerError(\n        "Usage: sle_transition <n_records> <root_bits> <block_bits> "\n        "<bucket_capacity> <epoch> <fault_mode>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\nFAULT_MODE = int(program.args[6])\n\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\nif FAULT_MODE not in (0, 1, 2, 3):\n    raise CompilerError("fault_mode must be 0, 1, 2, or 3")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\n# No single party supplies the complete epoch key. Security holds as long as\n# at least one contribution remains unknown to the adversary.\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\n\n# Deterministic public round constants. They are not keys and can be included\n# in the paper/implementation. MiMC computations remain over secret shares.\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp(value):\n    """Keyed MiMC-style permutation evaluated without opening key or input."""\n\n    x = value + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\nstart_timer(3)\n\nnew_db_locators = sint.Array(N_RECORDS)\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    new_db_locators[row] = secure_locator_prp(old_db_locators[row])\n\nfor position in range(N_INDEX_SLOTS):\n    new_index_locators[position] = secure_locator_prp(\n        old_index_locators[position]\n    )\n    new_index_valid[position] = index_valid[position]\n\n\n# Controlled active faults validate that the hidden relation, not merely the\n# record count, is checked before commit.\nif FAULT_MODE == 1:\n    new_index_valid[0] = sint(0)\nelif FAULT_MODE == 2:\n    new_index_locators[0] = new_index_locators[0] + 1\nelif FAULT_MODE == 3:\n    # Move the semantic element to a different public bucket by swapping two\n    # locator values while retaining the original valid flags.\n    temporary = new_index_locators[0]\n    new_index_locators[0] = new_index_locators[BUCKET_CAPACITY]\n    new_index_locators[BUCKET_CAPACITY] = temporary\n\n\n# The challenge is revealed only after all protected candidate values exist.\n# It reveals no database value. Two independent fingerprints reduce accidental\n# algebraic collisions in this executable prototype.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_1 = sint(1)\nexpected_2 = sint(1)\n\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        element_1 = locator + alpha_1 * block + beta_1 * bucket\n        element_2 = locator + alpha_2 * block + beta_2 * bucket\n        expected_1 *= challenge_1 - element_1\n        expected_2 *= challenge_2 - element_2\n\nactual_1 = sint(1)\nactual_2 = sint(1)\n\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n\n    element_1 = locator + alpha_1 * block + beta_1 * bucket\n    element_2 = locator + alpha_2 * block + beta_2 * bucket\n    factor_1 = valid * (challenge_1 - element_1) + (1 - valid)\n    factor_2 = valid * (challenge_2 - element_2) + (1 - valid)\n    actual_1 *= factor_1\n    actual_2 *= factor_2\n\nrelation_ok = (expected_1 == actual_1) * (expected_2 == actual_2)\n\n# Atomic selection happens while values remain shared. A rejected transition\n# writes the old state; an accepted transition writes the evolved state.\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\n# MP-SPDZ writes each party\'s share, not a reconstructed locator. The files can\n# be loaded by a later MPC epoch without ever assembling the protected state.\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\n\nstop_timer(3)\n\nprint_ln("SLE_MPC_RESULT mode=transition epoch=%s relation_ok=%s fault=%s",\n         NEW_EPOCH, opened_relation_ok, FAULT_MODE)\nprint_ln("SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n         "mapping=not_materialized")\n\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
from pathlib import Path
target = Path('/content/sle_mpc_project/scripts/generate_inputs.py')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Generate reproducible MP-SPDZ inputs for the SLE MPC programs.\n\nThe default path creates a small synthetic gallery/probe pair so the\ncryptographic layer can be tested without downloading a face dataset. An NPZ\nexport from the biometric notebook can be supplied later; the MPC programs see\nonly binary roots, never face images or real-valued FaceNet embeddings.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n\nSEED = 20260720\n\n\n@dataclass\nclass ProtectedInputs:\n    gallery: np.ndarray\n    labels: np.ndarray\n    tie_tags: np.ndarray\n    probe: np.ndarray\n    query_label: int\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", choices=("full_scan", "oram", "transition"), required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--n-records", type=int, default=16)\n    parser.add_argument("--root-bits", type=int, default=64)\n    parser.add_argument("--block-bits", type=int, default=4)\n    parser.add_argument("--bucket-capacity", type=int, default=8)\n    parser.add_argument("--query-row", type=int, default=0)\n    parser.add_argument("--flip-probability", type=float, default=0.05)\n    parser.add_argument("--npz", type=Path)\n    return parser.parse_args()\n\n\ndef load_or_generate(args: argparse.Namespace) -> ProtectedInputs:\n    rng = np.random.default_rng(SEED)\n\n    if args.npz is not None:\n        archive = np.load(args.npz)\n        gallery = np.asarray(archive["gallery_roots"], dtype=np.int64)\n        labels = np.asarray(\n            archive.get("labels", np.arange(len(gallery))), dtype=np.int64\n        )\n        probes = np.asarray(archive["probe_roots"], dtype=np.int64)\n        if gallery.ndim != 2 or probes.ndim != 2:\n            raise ValueError("gallery_roots and probe_roots must be two-dimensional")\n        if gallery.shape != probes.shape:\n            raise ValueError("gallery_roots and probe_roots must have equal shape")\n        if len(gallery) < args.n_records or gallery.shape[1] < args.root_bits:\n            raise ValueError("NPZ does not contain enough records/root bits")\n        gallery = gallery[: args.n_records, : args.root_bits]\n        labels = labels[: args.n_records]\n        probe = probes[args.query_row, : args.root_bits]\n    else:\n        gallery = rng.integers(\n            0, 2, size=(args.n_records, args.root_bits), dtype=np.int64\n        )\n        labels = np.arange(1000, 1000 + args.n_records, dtype=np.int64)\n\n        # Guarantee that the first flattened index slot is real. This is used\n        # only by the explicit fault-injection smoke tests.\n        gallery[0, : args.block_bits] = 0\n\n        probe = gallery[args.query_row].copy()\n        flips = rng.random(args.root_bits) < args.flip_probability\n        probe[flips] ^= 1\n\n    if not np.isin(gallery, (0, 1)).all() or not np.isin(probe, (0, 1)).all():\n        raise ValueError("MPC roots must be binary")\n\n    tie_tags = rng.permutation(np.arange(1, args.n_records + 1, dtype=np.int64))\n    return ProtectedInputs(\n        gallery=gallery,\n        labels=labels,\n        tie_tags=tie_tags,\n        probe=probe,\n        query_label=int(labels[args.query_row]),\n    )\n\n\ndef block_value(root: np.ndarray, block: int, block_bits: int) -> int:\n    start = block * block_bits\n    value = 0\n    for bit, item in enumerate(root[start : start + block_bits]):\n        value |= int(item) << bit\n    return value\n\n\ndef build_bucket_rows(\n    gallery: np.ndarray,\n    block_bits: int,\n    capacity: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    n_records, root_bits = gallery.shape\n    if root_bits % block_bits:\n        raise ValueError("root_bits must be divisible by block_bits")\n\n    n_blocks = root_bits // block_bits\n    bucket_values = 2**block_bits\n    n_buckets = n_blocks * bucket_values\n    dummy_row = n_records\n    rows = np.full((n_buckets, capacity), dummy_row, dtype=np.int64)\n    valid = np.zeros((n_buckets, capacity), dtype=np.int64)\n    occupancy = np.zeros(n_buckets, dtype=np.int64)\n\n    for row in range(n_records):\n        for block in range(n_blocks):\n            value = block_value(gallery[row], block, block_bits)\n            address = block * bucket_values + value\n            slot = int(occupancy[address])\n            if slot >= capacity:\n                raise ValueError(\n                    f"bucket {address} requires more than {capacity} slots; "\n                    "increase --bucket-capacity"\n                )\n            rows[address, slot] = row\n            valid[address, slot] = 1\n            occupancy[address] += 1\n\n    return rows, valid\n\n\ndef flatten(*arrays: np.ndarray | list[int] | tuple[int, ...]) -> list[int]:\n    output: list[int] = []\n    for array in arrays:\n        output.extend(np.asarray(array, dtype=np.int64).reshape(-1).tolist())\n    return output\n\n\ndef write_inputs(output_dir: Path, values_by_player: dict[int, list[int]]) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    for player in range(5):\n        values = values_by_player.get(player, [])\n        path = output_dir / f"Input-P{player}-0"\n        path.write_text(" ".join(str(int(value)) for value in values) + "\\n")\n\n\ndef full_scan_inputs(data: ProtectedInputs) -> dict[int, list[int]]:\n    return {\n        0: flatten(data.labels, data.tie_tags, data.gallery),\n        1: flatten(data.probe),\n    }\n\n\ndef oram_inputs(\n    data: ProtectedInputs,\n    block_bits: int,\n    capacity: int,\n) -> tuple[dict[int, list[int]], np.ndarray, np.ndarray]:\n    rows, valid_slots = build_bucket_rows(data.gallery, block_bits, capacity)\n    n_records, root_bits = data.gallery.shape\n\n    labels = np.concatenate((data.labels, [-1]))\n    valid_records = np.concatenate((np.ones(n_records, dtype=np.int64), [0]))\n    tie_tags = np.concatenate((data.tie_tags, [2**31 - 1]))\n    gallery = np.vstack((data.gallery, np.zeros(root_bits, dtype=np.int64)))\n\n    return (\n        {\n            0: flatten(labels, valid_records, tie_tags, gallery, rows),\n            1: flatten(data.probe),\n        },\n        rows,\n        valid_slots,\n    )\n\n\ndef transition_inputs(\n    data: ProtectedInputs,\n    block_bits: int,\n    capacity: int,\n) -> tuple[dict[int, list[int]], np.ndarray, np.ndarray]:\n    rows, valid_slots = build_bucket_rows(data.gallery, block_bits, capacity)\n    rng = np.random.default_rng(SEED + 1)\n    # Generate only the required unique locators. Building\n    # np.arange(1, 2**31) would allocate roughly 16 GiB before sampling and\n    # can make a Colab runtime terminate with exit code 137.\n    unique_locators: set[int] = set()\n    while len(unique_locators) < len(data.gallery):\n        unique_locators.add(int(rng.integers(1, 2**31)))\n    old_db_locators = np.asarray(sorted(unique_locators), dtype=np.int64)\n    dummy_locators = rng.integers(\n        1, 2**31, size=rows.shape, dtype=np.int64\n    )\n    old_index_locators = dummy_locators.copy()\n    for address in range(rows.shape[0]):\n        for slot in range(rows.shape[1]):\n            if valid_slots[address, slot]:\n                old_index_locators[address, slot] = old_db_locators[\n                    rows[address, slot]\n                ]\n\n    # Each party contributes one independent key component. The MPC program\n    # adds them while shared, so no party observes the complete epoch key.\n    contributions = rng.integers(1, 2**30, size=5, dtype=np.int64)\n    player_values = {\n        0: flatten(\n            old_db_locators,\n            data.gallery,\n            old_index_locators,\n            valid_slots,\n            [contributions[0]],\n        )\n    }\n    for player in range(1, 5):\n        player_values[player] = [int(contributions[player])]\n    return player_values, rows, valid_slots\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.n_records < 2:\n        raise ValueError("--n-records must be at least two")\n    if args.root_bits % args.block_bits:\n        raise ValueError("--root-bits must be divisible by --block-bits")\n\n    data = load_or_generate(args)\n    metadata: dict[str, object] = {\n        "mode": args.mode,\n        "n_records": args.n_records,\n        "root_bits": args.root_bits,\n        "block_bits": args.block_bits,\n        "bucket_capacity": args.bucket_capacity,\n        "query_row": args.query_row,\n        "expected_identity": data.query_label,\n        "plain_query_distance": int(\n            np.count_nonzero(data.gallery[args.query_row] != data.probe)\n        ),\n    }\n\n    if args.mode == "full_scan":\n        player_values = full_scan_inputs(data)\n    elif args.mode == "oram":\n        player_values, rows, valid = oram_inputs(\n            data, args.block_bits, args.bucket_capacity\n        )\n        metadata.update(\n            padded_slots=int(rows.size),\n            real_slots=int(valid.sum()),\n            maximum_occupancy=int(valid.sum(axis=1).max()),\n        )\n    else:\n        player_values, rows, valid = transition_inputs(\n            data, args.block_bits, args.bucket_capacity\n        )\n        metadata.update(\n            padded_slots=int(rows.size),\n            real_relation_entries=int(valid.sum()),\n            maximum_occupancy=int(valid.sum(axis=1).max()),\n        )\n\n    write_inputs(args.output_dir, player_values)\n    (args.output_dir / "sle_input_metadata.json").write_text(\n        json.dumps(metadata, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(metadata, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
from pathlib import Path
target = Path('/content/sle_mpc_project/scripts/verify_outputs.py')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Check the intentionally small public output of an MP-SPDZ SLE run."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nfrom pathlib import Path\n\n\nRESULT_RE = re.compile(\n    r"SLE_MPC_RESULT mode=(?P<mode>\\w+).*?"\n    r"(?:accepted=(?P<accepted>[01]).*?identity=(?P<identity>-?\\d+)|"\n    r"relation_ok=(?P<relation_ok>[01]))"\n)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--log", type=Path, required=True)\n    parser.add_argument("--metadata", type=Path)\n    parser.add_argument("--expect-relation", type=int, choices=(0, 1))\n    args = parser.parse_args()\n\n    text = args.log.read_text(encoding="utf-8", errors="replace")\n    match = RESULT_RE.search(text)\n    if match is None:\n        raise SystemExit("No SLE_MPC_RESULT line found")\n\n    result = match.groupdict()\n    if result["mode"] in ("full_scan", "oram"):\n        if args.metadata is None:\n            raise SystemExit("--metadata is required for identification modes")\n        metadata = json.loads(args.metadata.read_text(encoding="utf-8"))\n        expected = int(metadata["expected_identity"])\n        observed = int(result["identity"])\n        accepted = int(result["accepted"])\n        if accepted != 1 or observed != expected:\n            raise SystemExit(\n                f"Identification failed: expected {expected}, observed {observed}, "\n                f"accepted={accepted}"\n            )\n        print(f"PASS: {result[\'mode\']} opened only the expected identity {expected}")\n    else:\n        expected = args.expect_relation\n        observed = int(result["relation_ok"])\n        if expected is None:\n            raise SystemExit("--expect-relation is required for transition mode")\n        if observed != expected:\n            raise SystemExit(\n                f"Transition check failed: expected {expected}, observed {observed}"\n            )\n        print(f"PASS: transition relation_ok={observed}")\n\n\nif __name__ == "__main__":\n    main()\n\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# CELL 8. Install sources into MP-SPDZ
import shutil
project = Path('/content/sle_mpc_project')
mpspdz = Path('/content/MP-SPDZ')
for name in ('sle_full_scan.mpc', 'sle_oram_index.mpc', 'sle_transition.mpc'):
    shutil.copy2(project / 'Programs/Source' / name, mpspdz / 'Programs/Source' / name)
print('MP-SPDZ sources installed')


## Test 1: real MPC full scan

Player 0 inputs the gallery; player 1 inputs the probe. Five Shamir parties compute every Hamming distance and ArgMin. Only identity/reject is opened.

In [ ]:
# CELL 9. Generate full-scan inputs, compile, run five parties, verify
cfg = CFG
inputs = mpspdz / 'Player-Data'
run(f"python3 {project}/scripts/generate_inputs.py --mode full_scan "
    f"--output-dir {inputs} --n-records {cfg['n']} --root-bits {cfg['root_bits']} "
    f"--block-bits {cfg['block_bits']} --bucket-capacity {cfg['capacity']}")
program_name = f"sle_full_scan-{cfg['n']}-{cfg['root_bits']}-{cfg['threshold']}"
run(f"python3 compile.py -M sle_full_scan {cfg['n']} {cfg['root_bits']} {cfg['threshold']}", cwd=mpspdz)
env = os.environ.copy(); env.update(PLAYERS='5', THRESHOLD='2')
log = project / 'full_scan.log'
run(f"Scripts/shamir.sh {program_name}", cwd=mpspdz, env=env, log_path=log)
run(f"python3 {project}/scripts/verify_outputs.py --log {log} "
    f"--metadata {inputs}/sle_input_metadata.json")


## Test 2: real secret-index ORAM retrieval

The query-derived bucket address remains secret. Candidate row IDs are read from one ORAM and their roots from a second ORAM. The smoke profile uses linear ORAM; larger compile-time tables automatically select the recursive tree backend.

In [ ]:
# CELL 10. Generate ORAM inputs, compile with memory ordering, run, verify
run(f"python3 {project}/scripts/generate_inputs.py --mode oram "
    f"--output-dir {inputs} --n-records {cfg['n']} --root-bits {cfg['root_bits']} "
    f"--block-bits {cfg['block_bits']} --bucket-capacity {cfg['capacity']}")
program_name = (f"sle_oram_index-{cfg['n']}-{cfg['root_bits']}-"
                f"{cfg['block_bits']}-{cfg['capacity']}-{cfg['threshold']}")
run(f"python3 compile.py -M sle_oram_index {cfg['n']} {cfg['root_bits']} "
    f"{cfg['block_bits']} {cfg['capacity']} {cfg['threshold']}", cwd=mpspdz)
log = project / 'oram.log'
run(f"Scripts/shamir.sh {program_name}", cwd=mpspdz, env=env, log_path=log)
run(f"python3 {project}/scripts/verify_outputs.py --log {log} "
    f"--metadata {inputs}/sle_input_metadata.json")


## Test 3: SLE transition and integrity faults

Every party contributes to the epoch key. Clean transition must return `relation_ok=1`; drop, locator corruption, and wrong-bucket corruption must return `relation_ok=0`. Committed values are written as shares, not as reconstructed locators. This is the heaviest smoke-test cell.

In [ ]:
# CELL 11. Clean transition and three active-fault tests
run(f"python3 {project}/scripts/generate_inputs.py --mode transition "
    f"--output-dir {inputs} --n-records {cfg['n']} --root-bits {cfg['root_bits']} "
    f"--block-bits {cfg['block_bits']} --bucket-capacity {cfg['capacity']}")
transition_rows = []
for fault_mode in (0, 1, 2, 3):
    program_name = (f"sle_transition-{cfg['n']}-{cfg['root_bits']}-"
                    f"{cfg['block_bits']}-{cfg['capacity']}-42-{fault_mode}")
    _, compile_seconds = run(
        f"python3 compile.py -M sle_transition {cfg['n']} {cfg['root_bits']} "
        f"{cfg['block_bits']} {cfg['capacity']} 42 {fault_mode}", cwd=mpspdz)
    log = project / f'transition_fault_{fault_mode}.log'
    _, run_seconds = run(f"Scripts/shamir.sh {program_name}", cwd=mpspdz, env=env, log_path=log)
    expected = 1 if fault_mode == 0 else 0
    run(f"python3 {project}/scripts/verify_outputs.py --log {log} --expect-relation {expected}")
    transition_rows.append(dict(fault_mode=fault_mode, compile_seconds=compile_seconds, run_seconds=run_seconds))
import pandas as pd
display(pd.DataFrame(transition_rows))


## Optional: use real CelebA/FaceNet roots

Upload an NPZ containing `gallery_roots`, `probe_roots`, and optionally `labels`. Both root arrays must be binary with shape `[N, root_bits]`. Pass `--npz /content/your_file.npz` to the generator commands. Raw images and float embeddings are not supplied to MPC.

In [ ]:
# CELL 12. Collect concise paper-facing runtime/communication lines
import re
records = []
for path in sorted(project.glob('*.log')):
    text = path.read_text(encoding='utf-8', errors='replace')
    result = re.findall(r'SLE_MPC_RESULT[^\n]+', text)
    times = re.findall(r'Time = ([0-9.]+)', text)
    sent = re.findall(r'Data sent = ([0-9.eE+-]+)', text)
    records.append(dict(log=path.name, result=result[-1] if result else None,
                        runtime_seconds=times[-1] if times else None,
                        data_sent=sent[-1] if sent else None))
display(pd.DataFrame(records))
print('Interpretation: these are localhost five-party measurements, not WAN latency.')
